# 평가 프로토콜 — Hold-out F1 측정 (GPU 필요)

`pipeline.ipynb`는 테스트셋 정답이 비공개라 정량 평가가 없었습니다. 이 노트북은 그 공백을 메우기 위한
**재현 가능한 평가 프로토콜**입니다.

**아이디어.** 약한 라벨(silver label)이 부여된 학습 문서를 `train`/`validation`으로 분리하고,
`train`으로 모델을 학습한 뒤 `validation`에서 **Micro/Macro F1, Precision, Recall, Precision@3**을 측정합니다.

> ⚠️ **정직한 한계.** 검증 라벨도 키워드 매칭으로 만든 *약한 라벨*이므로, 이 점수는
> "사람이 만든 정답 대비 정확도"가 아니라 **"모델이 약한 감독 패턴을 얼마나 잘 일반화하는가"** 를 측정합니다.
> 그럼에도 (1) 학습 수렴/과적합 점검, (2) 하이퍼파라미터·구조 비교, (3) 상대적 개선 추적에는 유효한 지표입니다.

**실행 환경.** GPU 권장(BERT fine-tuning). CPU에서는 아래 `CONFIG['subset']`을 작게 잡아 스모크 테스트만 하세요.

## 0. 설정 & 라이브러리

In [ ]:
import os, re, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

CONFIG = {
    'model_name': 'bert-base-uncased',
    'batch_size': 32,
    'max_len': 128,
    'lr': 2e-5,
    'epochs': 3,
    'val_ratio': 0.15,     # 약한 라벨 데이터 중 검증셋 비율
    'threshold': 0.5,      # multi-label 판정 임계값 (sigmoid)
    'subset': None,        # CPU 스모크 테스트 시 정수(예: 1000)로 설정, 전체는 None
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CONFIG['seed'])
print("device:", CONFIG['device'])

## 1. 데이터 로딩 & 약한 라벨 생성
`pipeline.ipynb`와 동일한 로더·라벨링 로직을 사용합니다.

In [ ]:
ROOT = Path("Amazon_products")

def load_classes(path):
    id2name, name2id = {}, {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) >= 2:
                id2name[int(p[0])] = p[1]; name2id[p[1]] = int(p[0])
    return id2name, name2id

def load_hierarchy(path):
    parents = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) >= 2: parents[int(p[1])].append(int(p[0]))
    return parents

def load_keywords(path, name2id):
    ck = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if ":" in line:
                cname, kw = line.strip().split(":", 1)
                if cname in name2id:
                    ck[name2id[cname]] = [k.strip() for k in kw.split(",")]
    return ck

def load_corpus(path):
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.strip().split("\t", 1)
            if len(p) == 2: data.append((int(p[0]), p[1]))
    return sorted(data, key=lambda x: x[0])

id2name, name2id = load_classes(ROOT / "classes.txt")
parents_map = load_hierarchy(ROOT / "class_hierarchy.txt")
class_keywords = load_keywords(ROOT / "class_related_keywords.txt", name2id)
train_raw = load_corpus(ROOT / "train" / "train_corpus.txt")
num_classes = len(id2name)

def generate_silver_labels(corpus, class_keywords, parents_map, num_classes):
    compiled = {cid: re.compile(r"\b(" + "|".join(map(re.escape, kws)) + r")\b")
                for cid, kws in class_keywords.items()}
    labeled = []
    for did, text in tqdm(corpus, desc="silver labeling"):
        tl = text.lower()
        matched = {cid for cid, pat in compiled.items() if pat.search(tl)}
        if not matched:
            continue
        queue, visited = list(matched), set(matched)
        while queue:
            cur = queue.pop(0)
            for p in parents_map.get(cur, []):
                if p not in visited: visited.add(p); queue.append(p)
        vec = np.zeros(num_classes, dtype=np.float32)
        for c in visited: vec[c] = 1.0
        labeled.append((text, vec))
    return labeled

labeled = generate_silver_labels(train_raw, class_keywords, parents_map, num_classes)
if CONFIG['subset']:
    labeled = labeled[:CONFIG['subset']]
print("labeled docs:", len(labeled))

## 2. Train / Validation 분리

In [ ]:
rng = np.random.RandomState(CONFIG['seed'])
idx = rng.permutation(len(labeled))
n_val = int(len(labeled) * CONFIG['val_ratio'])
val_idx, train_idx = set(idx[:n_val].tolist()), set(idx[n_val:].tolist())
train_data = [labeled[i] for i in train_idx]
val_data   = [labeled[i] for i in val_idx]
print(f"train={len(train_data)}  val={len(val_data)}")

## 3. 모델 & 데이터셋
`pipeline.ipynb`의 `TaxoClassModel`/`BertDataset`과 동일합니다.

In [ ]:
class TaxoClassModel(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        self.dropout = nn.Dropout(0.1)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out[0][:, 0, :]
        return self.classifier(self.dropout(cls))

class BertDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data, self.tok, self.max_len = data, tokenizer, max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        text, labels = self.data[i]
        enc = self.tok.encode_plus(str(text), add_special_tokens=True, max_length=self.max_len,
                                   padding='max_length', truncation=True,
                                   return_attention_mask=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'labels': torch.tensor(labels, dtype=torch.float)}

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = TaxoClassModel(CONFIG['model_name'], num_classes).to(CONFIG['device'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
criterion = nn.BCEWithLogitsLoss()
train_loader = DataLoader(BertDataset(train_data, tokenizer, CONFIG['max_len']),
                          batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(BertDataset(val_data, tokenizer, CONFIG['max_len']),
                        batch_size=CONFIG['batch_size'] * 2, shuffle=False)

## 4. 평가 함수
Micro/Macro F1·Precision·Recall(임계값 기반)과 Precision@3/Recall@3(top-k 기반)을 계산합니다.

In [ ]:
def precision_recall_at_k(probs, y_true, k=3):
    topk = np.argsort(-probs, axis=1)[:, :k]
    ps, rs = [], []
    for i in range(len(probs)):
        true_set = set(np.where(y_true[i] > 0.5)[0])
        if not true_set: continue
        pred_set = set(topk[i].tolist())
        inter = len(true_set & pred_set)
        ps.append(inter / k); rs.append(inter / len(true_set))
    return float(np.mean(ps)), float(np.mean(rs))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_true = [], []
    for batch in loader:
        ids = batch['input_ids'].to(CONFIG['device'])
        mask = batch['attention_mask'].to(CONFIG['device'])
        logits = model(ids, attention_mask=mask)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_true.append(batch['labels'].numpy())
    probs = np.vstack(all_probs); y_true = np.vstack(all_true)
    y_pred = (probs >= CONFIG['threshold']).astype(int)
    res = {
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'micro_precision': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'micro_recall': recall_score(y_true, y_pred, average='micro', zero_division=0),
    }
    res['p@3'], res['r@3'] = precision_recall_at_k(probs, y_true, k=3)
    return res

## 5. 학습 + 에폭별 검증
학습 손실과 검증 지표 곡선을 함께 추적합니다.

In [ ]:
history = {'loss': [], 'micro_f1': [], 'macro_f1': [], 'p@3': []}
for epoch in range(CONFIG['epochs']):
    model.train(); running = 0.0
    for batch in tqdm(train_loader, desc=f"epoch {epoch+1}"):
        ids = batch['input_ids'].to(CONFIG['device'])
        mask = batch['attention_mask'].to(CONFIG['device'])
        labels = batch['labels'].to(CONFIG['device'])
        optimizer.zero_grad()
        loss = criterion(model(ids, attention_mask=mask), labels)
        loss.backward(); optimizer.step()
        running += loss.item()
    avg_loss = running / len(train_loader)
    val = evaluate(model, val_loader)
    history['loss'].append(avg_loss)
    for k in ['micro_f1', 'macro_f1', 'p@3']: history[k].append(val[k])
    print(f"epoch {epoch+1}: loss={avg_loss:.4f}  micro_f1={val['micro_f1']:.4f}  "
          f"macro_f1={val['macro_f1']:.4f}  P@3={val['p@3']:.4f}  R@3={val['r@3']:.4f}")

## 6. 최종 지표 & 학습 곡선

In [ ]:
final = evaluate(model, val_loader)
print("=== Validation metrics (held-out silver labels) ===")
for k, v in final.items():
    print(f"{k:16s}: {v:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, CONFIG['epochs'] + 1)
ax[0].plot(ep, history['loss'], marker='o', color='#C44E52'); ax[0].set_title("Training loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("BCE loss")
ax[1].plot(ep, history['micro_f1'], marker='o', label='micro F1')
ax[1].plot(ep, history['macro_f1'], marker='s', label='macro F1')
ax[1].plot(ep, history['p@3'], marker='^', label='P@3')
ax[1].set_title("Validation metrics"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); Path("figures").mkdir(exist_ok=True)
plt.savefig("figures/05_eval_curves.png", dpi=110); plt.show()

# (선택) 체크포인트 저장 — pipeline 의 영속화 부재 한계를 보완
# torch.save(model.state_dict(), "model_checkpoint.pt")

## 7. 해석 가이드

- **Micro F1** : 빈도가 높은(head) 클래스 위주의 종합 성능. 전반적 수렴 확인용.
- **Macro F1** : 클래스별 F1의 단순 평균 → **희소(tail) 클래스까지 잘 맞히는지**를 드러냄(낮으면 long-tail 문제).
- **P@3 / R@3** : 제출 전략(top-3)과 직접 대응되는 지표.
- micro F1은 높은데 macro F1이 낮다면 → EDA에서 본 **클래스 쏠림**이 그대로 재현되는 것.

**활용.** 인코더(`bert-base` → `roberta`/`deberta`), `max_len`, 임계값, self-training 반복 수 등을 바꿔가며
이 노트북의 지표로 **변화량을 정량 비교**하면, "정확도를 올리는 실험"을 근거 있게 진행할 수 있습니다.